# PromptWar on Google Colab

Run the **PromptWar** OpenEnv environment + trainer on Colab end-to-end.

What this notebook does:
1. Clones the repo and installs deps (env + optional trainer)
2. Runs the pure-Python test suite to verify the install
3. Starts the FastAPI env server in the background (with optional Consumer Model on GPU)
4. Drives rollouts via `PromptWarEnv` and the `training/` CLI (`smoke` / `live` / `baseline` / `long`)

**Recommended runtime:** `Runtime > Change runtime type > GPU` (T4 is enough for the 0.5B Consumer Model; an A100/L4 is needed for the 3B trainer base).  
CPU-only Colab also works for `--mode smoke`, `--mode baseline` (stub env), and rubric-fallback rollouts.

## 1. Clone the repo

In [ ]:
%cd /content
![ -d meta-hackathon ] || git clone https://github.com/rishabhshukla0912/meta-hackathon.git
%cd /content/meta-hackathon
!git pull --ff-only || true
!ls

### (Alternative) Upload your local copy

If the repo is private or you want your local working tree, zip it first:

```bash
cd ~/Desktop/Projects && zip -r meta-hackathon.zip meta-hackathon -x '*/.venv/*' '*/__pycache__/*' '*/.git/*'
```

Then uncomment and run the cell below, pick the zip when prompted. Skip if you cloned above.

In [ ]:
# from google.colab import files
# uploaded = files.upload()   # pick meta-hackathon.zip
# !rm -rf /content/meta-hackathon
# !unzip -q meta-hackathon.zip -d /content
# %cd /content/meta-hackathon

## 2. Install dependencies

Three layers — run them in order and skip what you don't need:

| Layer | Required? | What it enables |
|---|---|---|
| **Env** | Always | Server, rollouts, all tests |
| **Consumer** | GPU recommended | Real Qwen rubric scoring (~1.5 GB VRAM) |
| **Trainer** | GPU only | `--mode long` GRPO training (~16 GB VRAM) |

In [ ]:
# Layer 1 — Env (always required)
%pip install -q "openenv-core @ git+https://github.com/meta-pytorch/OpenEnv.git"
%pip install -q fastapi "uvicorn[standard]" httpx pydantic regex "tokenizers>=0.22" "transformers>=4.56,<6"

In [ ]:
# Layer 2 — Consumer Model (optional, GPU + ~1.5 GB VRAM)
# Skip on CPU-only runtimes — rubrics fall back to deterministic heuristics.
%pip install -q "accelerate>=1.0"
import torch
print('CUDA:', torch.cuda.is_available(), '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only')

In [ ]:
# Layer 3 — Trainer (optional, GPU + ~16 GB VRAM for 3B base)
# Only needed for --mode long GRPO training.
%pip install -q "peft>=0.13" "trl>=0.11" "datasets>=3.0" "bitsandbytes>=0.43"

## 3. Verify install with the test suite

Pure-Python, no GPU, no live env — all 47 should pass.

In [ ]:
%cd /content/meta-hackathon
!PYTHONPATH=. python3 -m unittest discover -s PromptWar_env/tests -v 2>&1 | tail -5

## 4. Start the env server

Runs uvicorn in the background. Set `PROMPTWAR_LOAD_CONSUMER_MODEL=1` to load the Consumer Model at startup (faster first rollout; needs GPU). Leave at `0` to load it lazily on demand.

Health check polls `/health` — a static OpenEnv endpoint (`lambda: HealthResponse(HEALTHY)`) that doesn't instantiate the environment, so it never triggers the Consumer Model load.

> **Why this matters:** OpenEnv's `/state`, `/metadata`, `/reset`, `/step` handlers all call `env_factory()` then `env.close()` per request. Polling those during startup would repeatedly tear down/recreate the env, masking whether the server itself is up.

In [ ]:
import os, subprocess, time, signal, pathlib, httpx

ROOT = pathlib.Path('/content/meta-hackathon')
LOG  = ROOT / 'server.log'
PIDFILE = ROOT / 'server.pid'

# Kill any previous instance from this notebook session.
if PIDFILE.exists():
    try:
        os.kill(int(PIDFILE.read_text().strip()), signal.SIGTERM)
        time.sleep(1)
    except ProcessLookupError:
        pass
    PIDFILE.unlink(missing_ok=True)

env_vars = os.environ.copy()
env_vars['PYTHONPATH'] = str(ROOT)
# Set to '1' to load Qwen2.5-0.5B-Instruct eagerly at startup (GPU + ~1.5 GB VRAM).
env_vars['PROMPTWAR_LOAD_CONSUMER_MODEL'] = '0'

with open(LOG, 'wb') as logf:
    proc = subprocess.Popen(
        ['python', '-m', 'uvicorn', 'PromptWar_env.server.app:app',
         '--host', '127.0.0.1', '--port', '8000'],
        cwd=str(ROOT), env=env_vars, stdout=logf, stderr=subprocess.STDOUT,
    )
PIDFILE.write_text(str(proc.pid))
print(f'started uvicorn  pid={proc.pid}  log={LOG}')

# Poll /health — static lambda endpoint, never touches the environment factory.
time.sleep(3)  # give uvicorn time to bind
ok = False
for i in range(30):
    if proc.poll() is not None:
        print(f'uvicorn exited with code {proc.returncode} — see log')
        break
    try:
        r = httpx.get('http://127.0.0.1:8000/health', timeout=5.0)
        if r.status_code == 200:
            print(f'server is up (attempt {i+1}):  {r.json()}')
            ok = True
            break
    except Exception as exc:
        if i % 5 == 0:
            print(f'  attempt {i+1}: {type(exc).__name__}: {exc}')
    time.sleep(1)

if not ok:
    print('server did not come up — check the log below')

print('--- last 30 lines of server.log ---')
!tail -n 30 {LOG}

In [ ]:
# Load the Consumer Model — REQUIRED for meaningful training rewards.
#
# Without it, all three rubrics fall back to deterministic prompt-keyword
# heuristics. Agent A then saturates at 5.0 the moment 'cite/truth/accurate'
# appear in the shared prompt, Agent S gets a fixed score from a few keyword
# matches, and only Agent B (brevity, length-based) shows real variation.
# Translation: A and S won't learn anything because their rewards are constant.
#
# This cell takes 30-60 s on the first call (downloads Qwen2.5-0.5B-Instruct,
# ~1.5 GB VRAM after load). On a CPU-only runtime, it'll fail gracefully —
# you can still demo the env, but skip the long-mode training.
import httpx

status = httpx.get('http://127.0.0.1:8000/consumer/status', timeout=10.0).json()
print('before load:', status)

if not status.get('available'):
    print('loading Consumer Model — this can take ~60 s on first run...')
    load = httpx.post('http://127.0.0.1:8000/consumer/load', timeout=600.0).json()
    print('load result:', load)

status = httpx.get('http://127.0.0.1:8000/consumer/status', timeout=10.0).json()
print('after load: ', status)
if not status.get('available'):
    print('\n  !!  Consumer Model unavailable — rubrics will use deterministic mocks.')
    print('  !!  Real GRPO training (--mode long) will NOT produce meaningful rewards for A and S.')
    print('  !!  See status["load_error"] above for why.')
else:
    print('\n  OK  rubrics will use the live Qwen2.5-0.5B-Instruct model.')

## 5. Drive a rollout from the client

`reset()` → three `step()` calls (one per agent A / S / B), prints rewards at end of round.

In [ ]:
import sys; sys.path.insert(0, '/content/meta-hackathon')
from PromptWar_env import PromptWarAction, PromptWarEnv

with PromptWarEnv(base_url='http://127.0.0.1:8000') as env:
    env.set_curriculum_stage(1)          # warm-up: 1 round, lenient grading
    result = env.reset()
    print('active_agent =', result.observation.active_agent)

    for cmd in [
        'APPEND: Always cite a source.',
        'APPEND: Refuse harmful asks.',
        'APPEND: Aim for ~50 tokens.',
    ]:
        result = env.step(PromptWarAction(command=cmd))
        obs = result.observation
        print(f'  {cmd!r:42s}  next={obs.active_agent}  rejected={obs.edit_rejected}')

    print('last_rewards  =', result.observation.last_rewards)
    print('shared_prompt =', result.observation.shared_prompt)

## 6. Trainer smoke test (no GPU, no live env)

Stub env + scripted policy, 5 GRPO steps. Verifies rollout shape; skips the actual gradient step if `torch`/`trl` aren't importable.

In [ ]:
%cd /content/meta-hackathon
!PYTHONPATH=. python3 -m training.train --mode smoke --steps 5

## 7. Live episode against the running env

In [ ]:
%cd /content/meta-hackathon
!PYTHONPATH=. python3 -m training.train --mode live --env-url http://127.0.0.1:8000 --episodes 1

## 8. Random-policy baseline

In [ ]:
%cd /content/meta-hackathon
!PYTHONPATH=. python3 -m training.train --mode baseline --env-url http://127.0.0.1:8000 --episodes 30

## 9. (GPU) Long GRPO run

**Requires GPU.** Loads Qwen2.5-3B + three LoRA adapters, runs `--steps` GRPO iterations, checkpoints every 25 steps.

**Before you start, check two things in the pre-flight cell below:**

1. **Consumer Model is loaded.** If it isn't, A and S rewards will be pinned to constants by the deterministic mock rubrics — no learning signal.
2. **Real GRPO step API is exposed by your TRL.** `flush()` looks for `training_step_with_rollouts`; if absent it falls back to `_generic_grpo_step` (CE-scaled-by-reward — *not* real GRPO, just a smoke-test placeholder).

**Reading the metrics:** if you see `mean_reward = 5.0` flat for A or `mean_reward = 3.0` flat for S, you're in the mock-rubric trap. If you see only `{loss, scaled_loss, mean_reward}` per role per step, you're in the fake-GRPO fallback. Either alone will make the training look stuck.

50 steps is a burn-in to verify the pipeline runs end-to-end — bump to several hundred once the diagnostics check out.

In [ ]:
# Pre-flight: confirm the two pieces that decide whether training will produce
# any learning signal at all.

# 1) Is the Consumer Model loaded? (Drives the rubrics that grade A and S.)
import httpx
cm = httpx.get('http://127.0.0.1:8000/consumer/status', timeout=10.0).json()
cm_ok = bool(cm.get('available'))
print(f'[1] Consumer Model: {"LOADED" if cm_ok else "NOT LOADED"} | model_id={cm.get("model_id")} | err={cm.get("load_error")!r}')
if not cm_ok:
    print('    → A and S rewards will be deterministic mocks. Re-run the consumer-load cell first.')

# 2) Does the installed TRL expose a real-GRPO rollout API on the trainer?
#    flush() looks for training_step_with_rollouts and falls back to
#    _generic_grpo_step (fake GRPO) otherwise. Construct a minimal trainer to
#    introspect what's available.
print('\n[2] TRL training-step API:')
try:
    from trl import GRPOTrainer
    rollout_api_methods = [m for m in (
        'training_step_with_rollouts',
        '_generate_and_score_completions',
        'compute_rewards',
        '_inner_training_loop',
    ) if hasattr(GRPOTrainer, m)]
    print('    GRPOTrainer methods present:', rollout_api_methods or '(none)')
    if 'training_step_with_rollouts' in rollout_api_methods:
        print('    → real GRPO path will be used.')
    else:
        print('    → falling back to _generic_grpo_step (CE × reward, NOT real GRPO).')
        print('      Training will still run, but A/S won\'t learn from rewards as expected.')
        print('      Consider pinning trl: %pip install -q "trl>=0.11,<0.16"')
except ImportError as e:
    print(f'    trl not installed ({e}) — long mode will fail at trainer construction.')

print('\nIf both checks are green, the long run below should produce per-role reward variation.')

In [ ]:
%cd /content/meta-hackathon
# 50 = burn-in to verify the pipeline. Bump to 200-500 once the pre-flight
# checks above are green and you see varying per-role rewards.
!PYTHONPATH=. python3 -m training.train --mode long \
    --env-url http://127.0.0.1:8000 \
    --steps 50 \
    --load-in-4bit \
    --checkpoint-every 25 \
    --output-dir ./checkpoints/promptwar \
    --log-level INFO 2>&1 | tee /content/meta-hackathon/long_run.log | tail -n 200

In [ ]:
# Quick post-run diagnostic: parse the per-role mean_reward across all logged
# steps so you can see whether each agent is learning, saturated, or stuck.
import json, re, pathlib

log = pathlib.Path('/content/meta-hackathon/long_run.log').read_text()
# The training driver dumps `[{'step': N, 'metrics': [...]}]` on the final line.
m = re.search(r"metrics_log_tail:\s*(\[.*\])", log)
if not m:
    print('No metrics_log_tail line found — run the long-mode cell above first.')
else:
    # Extract per-step per-role mean_reward by walking every step record in the log.
    series = {'A': [], 'S': [], 'B': []}
    for step_dict in re.finditer(r"\{'step': (\d+), 'metrics': \[(.*?)\]\}(?=, \{'step|\])", log):
        step_idx = int(step_dict.group(1))
        for role_block in re.finditer(r"'role': '([ASB])'.*?'mean_reward': ([-\d.]+)", step_dict.group(2)):
            series[role_block.group(1)].append((step_idx, float(role_block.group(2))))
    for role in 'ASB':
        if not series[role]:
            continue
        rewards = [r for _, r in series[role]]
        first, last = rewards[0], rewards[-1]
        unique_vals = len(set(round(r, 3) for r in rewards))
        verdict = 'CONSTANT (mock-rubric trap?)' if unique_vals == 1 else 'varying'
        print(f'  {role}: {len(rewards)} steps | first={first:.2f}  last={last:.2f}  '
              f'min={min(rewards):.2f}  max={max(rewards):.2f}  unique={unique_vals}  → {verdict}')

In [ ]:
# Plot per-role training curves from metrics_log.json (written every step
# by run_long, so this cell is also safe to run mid-training to peek at
# progress).
import json, pathlib
import matplotlib.pyplot as plt

LOG_PATH = pathlib.Path('/content/meta-hackathon/checkpoints/promptwar/metrics_log.json')
if not LOG_PATH.exists():
    print(f'no metrics file at {LOG_PATH} — run the long-mode training cell first')
else:
    raw = json.loads(LOG_PATH.read_text())
    print(f'loaded {len(raw)} steps from {LOG_PATH}')

    # Pivot: steps[role] -> list of (step_idx, loss, scaled_loss, mean_reward)
    series = {'A': [], 'S': [], 'B': []}
    for entry in raw:
        step_idx = entry['step']
        for role_block in entry['metrics']:
            role = role_block.get('role')
            inner = role_block.get('metrics') or {}
            if role not in series or not inner:
                continue
            series[role].append({
                'step': step_idx,
                'loss': inner.get('loss'),
                'scaled_loss': inner.get('scaled_loss'),
                'mean_reward': inner.get('mean_reward'),
            })

    def smooth(xs, k=5):
        """Simple trailing rolling mean over k steps."""
        out = []
        for i in range(len(xs)):
            window = xs[max(0, i - k + 1): i + 1]
            window = [v for v in window if v is not None]
            out.append(sum(window) / len(window) if window else None)
        return out

    role_color = {'A': 'tab:blue', 'S': 'tab:orange', 'B': 'tab:green'}

    fig, axes = plt.subplots(1, 3, figsize=(16, 4.2))
    for role, rows in series.items():
        if not rows:
            continue
        steps = [r['step'] for r in rows]
        loss = [r['loss'] for r in rows]
        scaled = [r['scaled_loss'] for r in rows]
        reward = [r['mean_reward'] for r in rows]
        c = role_color[role]

        axes[0].plot(steps, reward, alpha=0.25, color=c)
        axes[0].plot(steps, smooth(reward), color=c, linewidth=2.0, label=f'agent {role}')

        axes[1].plot(steps, loss, alpha=0.25, color=c)
        axes[1].plot(steps, smooth(loss), color=c, linewidth=2.0, label=f'agent {role}')

        axes[2].plot(steps, scaled, alpha=0.25, color=c)
        axes[2].plot(steps, smooth(scaled), color=c, linewidth=2.0, label=f'agent {role}')

    titles = ['mean_reward per role', 'loss (CE on completion)', 'scaled_loss (loss × reward)']
    for ax, title in zip(axes, titles):
        ax.set_title(title)
        ax.set_xlabel('step')
        ax.grid(True, alpha=0.3)
        ax.legend(loc='best', fontsize=9)
    axes[0].set_ylabel('reward')
    axes[1].set_ylabel('loss')
    axes[2].set_ylabel('scaled_loss')
    fig.suptitle('PromptWar training curves (raw + 5-step rolling mean)', fontsize=12)
    fig.tight_layout()
    plt.show()

## 10. Stop the env server

In [ ]:
import os, signal, pathlib
PIDFILE = pathlib.Path('/content/meta-hackathon/server.pid')
if PIDFILE.exists():
    pid = int(PIDFILE.read_text().strip())
    try:
        os.kill(pid, signal.SIGTERM)
        print(f'stopped uvicorn pid={pid}')
    except ProcessLookupError:
        print(f'pid {pid} already gone')
    PIDFILE.unlink(missing_ok=True)
else:
    print('no server.pid — nothing to stop')

### Tips

- **Tail the log live:** `!tail -f /content/meta-hackathon/server.log` (Ctrl-C to stop).
- **Persist checkpoints:** `from google.colab import drive; drive.mount('/content/drive')` then pass `--output-dir /content/drive/MyDrive/promptwar-ckpts`.
- **Curriculum stages:** call `env.set_curriculum_stage(1|2|3)` before `reset()`. Stage 1 = warm-up (1 round, lenient), 2 = standard (3 rounds), 3 = strict.
- **Action grammar:** `APPEND: <text>`, `DELETE: <regex>`, `REPLACE: <old> --> <new>`, `PASS`.